# AI Resume Screening & Job Category Prediction
## End-to-End NLP Project

This notebook builds a complete resume screening system using the provided **Resume dataset.csv**.

### Project objectives
- Explore and understand resume data
- Clean and preprocess resume text
- Convert text into TF-IDF numerical features
- Train an NLP classification model
- Evaluate the model using accuracy, classification report and confusion matrix
- Save the trained pipeline
- Build a user-facing Streamlit UI where a recruiter can upload/paste a resume
- Predict the most likely job category and show top matching categories

> **Important:** This is an educational resume classification project. It should assist recruiters rather than make final employment decisions automatically.


In [ ]:
# Install dependencies if needed
# !pip install pandas numpy scikit-learn matplotlib seaborn joblib streamlit pypdf python-docx openpyxl


In [ ]:
import os, re, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)

DATA_PATH = "data/Resume dataset.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())


## 1. Data Understanding

In [ ]:
print("Columns:", df.columns.tolist())
print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())
print("\nCategory distribution:")
display(df["category"].value_counts())


In [ ]:
plt.figure(figsize=(12,6))
sns.countplot(data=df, y="category", order=df["category"].value_counts().index)
plt.title("Resume Count by Job Category")
plt.xlabel("Number of Resumes")
plt.ylabel("Job Category")
plt.tight_layout()
plt.show()


## 2. Text Preprocessing

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\b\d{10,}\b", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["Text"].apply(clean_text)
display(df[["Text", "clean_text", "category"]].head(3))


In [ ]:
# Basic text-length analysis
df["text_length"] = df["clean_text"].str.len()
display(df.groupby("category")["text_length"].agg(["count","mean","min","max"]).sort_values("mean", ascending=False))

plt.figure(figsize=(10,5))
sns.histplot(df["text_length"], bins=50, kde=True)
plt.title("Distribution of Resume Text Length")
plt.xlabel("Characters")
plt.show()


## 3. Train/Test Split

We use a stratified split so that all job categories remain represented in both training and testing data.

In [ ]:
X = df["clean_text"]
y = df["category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))


## 4. TF-IDF + Linear SVM

**TF-IDF** gives higher importance to words that are useful for distinguishing categories.

**Linear SVM** is a strong baseline for high-dimensional text classification.

In [ ]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        sublinear_tf=True,
        max_features=50000,
        ngram_range=(1,2),
        min_df=2,
        max_df=0.95,
        stop_words="english"
    )),
    ("classifier", LinearSVC(C=1.5))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")


In [ ]:
print(classification_report(y_test, y_pred))


In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(12,9))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 5. Resume Prediction Function

In [ ]:
def predict_resume(resume_text, top_n=3):
    cleaned = clean_text(resume_text)
    prediction = model.predict([cleaned])[0]

    # LinearSVC decision scores
    scores = model.decision_function([cleaned])[0]
    classes = model.named_steps["classifier"].classes_
    ranking = np.argsort(scores)[::-1][:top_n]

    results = pd.DataFrame({
        "Category": classes[ranking],
        "Model Score": scores[ranking]
    })

    # Relative score only; not a calibrated probability.
    min_s, max_s = results["Model Score"].min(), results["Model Score"].max()
    if max_s != min_s:
        results["Relative Match %"] = (results["Model Score"] - min_s) / (max_s - min_s) * 100
    else:
        results["Relative Match %"] = 100.0

    return prediction, results

sample_resume = X_test.iloc[0]
predicted, ranking = predict_resume(sample_resume)
print("Predicted category:", predicted)
display(ranking)


## 6. Save the Model

The saved pipeline contains both TF-IDF preprocessing and the classifier, so the Streamlit app can use the exact same preprocessing used during training.

In [ ]:
os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/resume_classifier.joblib")

metrics = {
    "accuracy": float(accuracy),
    "classes": sorted(y.unique().tolist())
}
import json
with open("models/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved model and metrics.")


## 7. Example Predictions

Try a few short examples to understand how the classifier behaves.

In [ ]:
examples = {
    "Java Developer": "Java Spring Boot Hibernate REST API microservices Maven Git backend developer",
    "Web Developer": "HTML CSS JavaScript React frontend web development responsive websites",
    "SQL Developer": "SQL database queries stored procedures Oracle PL SQL database developer",
    "Project Manager": "project management agile scrum stakeholder management team delivery planning"
}

for name, text in examples.items():
    pred, top = predict_resume(text)
    print(f"\n{name} example -> {pred}")
    display(top)


# Final Result

The notebook creates an end-to-end NLP resume classification pipeline:

**CSV → Cleaning → TF-IDF → Linear SVM → Evaluation → Saved Model → Streamlit UI**

The next file in this project, `app.py`, provides the interactive recruiter-facing interface.
